# LCEL 빌딩 블록


## OpenAI LLM 준비

저급 모델 `gpt-4o-mini`일 때 출력 확인해보기


In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# .env 파일 로드
load_dotenv()

# LLM 준비
llm = init_chat_model(model="openai:gpt-4o", temperature=0.1)

print("✅ LLM 준비 완료")

## 프롬프트 메시지 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 명탐정 코난 영화 전문가입니다. 질문에 대해 명확하고 논리적으로 답하세요.",
        ),
        (
            "human",
            "{question} **{instruction}** 영화 제목과 개봉 연도만 간결하게 답변하세요.",
        ),
    ]
)

# 동일한 프롬프트에서 partial()로 변수를 다르게 바인딩
descending_prompt = prompt.partial(instruction="개봉 연도 내림차순으로 답변하세요.")
ascending_prompt = prompt.partial(instruction="개봉 연도 오름차순으로 답변하세요.")

print("✅ 프롬프트 템플릿 메시지 구성 완료 (partial 바인딩 적용)")

## LCEL 체인 구성

### 영화 개봉 연도 내림차순 출력


In [ ]:
from langchain_core.output_parsers import StrOutputParser

descending_chain = descending_prompt | llm | StrOutputParser()

descending_movies = descending_chain.invoke(
    {
        "question": "가장 최근에 영화로 개봉된 '명탐정 코난' 한국어 제목과 개봉 연도를 5개 알려주세요."
    }
)
print(f"✅ 명탐정 코난 영화 내림차순 정리: \n{descending_movies}")

### 영화 개봉 연도 오름차순 출력


In [ ]:
ascending_chain = ascending_prompt | llm | StrOutputParser()

ascending_movies = ascending_chain.invoke(
    {
        "question": "가장 최근에 영화로 개봉된 '명탐정 코난' 한국어 제목과 개봉 연도를 5개 알려주세요."
    }
)
print(f"✅ 명탐정 코난 영화 오름차순 정리: \n{ascending_movies}")

### RunnableParallel을 활용한 복합 체인 구성


In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    {
        "ascending_movies": ascending_chain,
        "descending_movies": descending_chain,
    }
)

results = parallel_chain.invoke(
    {
        "question": "가장 최근에 영화로 개봉된 '명탐정 코난' 한국어 제목과 개봉 연도를 5개 알려주세요."
    }
)

print(f"✅ 명탐정 코난 개봉 연도 오름차순:\n{results['ascending_movies']}")
print("--" * 20)
print(f"✅ 명탐정 코난 개봉 연도 내림차순:\n{results['descending_movies']}")